# CDR-MLC Causal Adaptive Feature Modulation

A leakage-safe ablation of four inference modes:

- frozen hard K-Means (baseline)
- causal adaptive centroids only
- frozen centroids plus cluster-conditioned robust feature modulation
- causal adaptive centroids plus robust feature modulation

Each block is predicted before its unlabeled observations update the routing/feature buffers. Test labels are accessed only after every prediction. Parameters are fixed before evaluation: block=2048, buffer=8192, minimum history=256, centroid learning rate=0.10, anchor=0.02, maximum modulation=0.75.


In [ ]:
import json
from collections import deque
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

MAIN = Path("CDR-MLC.ipynb")
if not MAIN.exists():
    MAIN = Path("CDR_MLC") / "CDR-MLC.ipynb"
namespace = {}
with MAIN.open(encoding="utf-8") as handle:
    notebook = json.load(handle)
exec(compile("".join(notebook["cells"][0]["source"]), str(MAIN), "exec"), namespace)
run_pipeline_from_two_files = namespace["run_pipeline_from_two_files"]
compute_sliding_window_stats = namespace["compute_sliding_window_stats"]
try:
    display
except NameError:
    display = print
print("Loaded leakage-safe CDR-MLC core")


In [ ]:
def _predict_aligned(classifier, X, number_of_classes):
    probability = classifier.predict_proba(X)
    aligned = np.zeros((len(X), number_of_classes))
    for column, label in enumerate(classifier.classes_):
        aligned[:, int(label)] = probability[:, column]
    return aligned.argmax(axis=1)


def evaluate_adaptive_modulation(
    scenario, train_file, test_file, block_size=2048, buffer_size=8192,
    minimum_history=256, center_learning_rate=0.10,
    center_anchor=0.02, maximum_modulation=0.75,
):
    result = run_pipeline_from_two_files(
        train_file, test_file, n_clusters=3, window_size=3,
        clustering_stats=["mean", "median", "std", "min", "max"],
    )
    train, test = result["train_df"], result["test_df"]
    features = result["classification_features"]
    X_test = test[features].to_numpy(float)
    experts = result["classifiers"]
    expert_ids = sorted(experts)
    number_of_classes = len(result["label_encoder"].classes_)

    routing_stats, _ = compute_sliding_window_stats(
        test[["SynAck", "AckDat", "TcpRtt"]],
        ["SynAck", "AckDat", "TcpRtt"], 3,
        ["mean", "median", "std", "min", "max"],
    )
    routing_vectors = result["scaler"].transform(routing_stats)
    original_centers = result["kmeans_model"].cluster_centers_.copy()

    # Robust per-expert training references.
    references = {}
    for expert_id in expert_ids:
        values = train.loc[train["cluster"] == expert_id, features].to_numpy(float)
        median = np.median(values, axis=0)
        robust_scale = np.maximum(
            (np.quantile(values, .75, axis=0) -
             np.quantile(values, .25, axis=0)) / 1.349,
            1e-9,
        )
        references[expert_id] = (
            median, robust_scale,
            np.quantile(values, .005, axis=0),
            np.quantile(values, .995, axis=0),
        )

    def predict_variant(adaptive_router, modulation):
        centers = original_centers.copy()
        feature_history = {k: deque() for k in expert_ids}
        route_history = {k: deque() for k in expert_ids}
        feature_sizes = {k: 0 for k in expert_ids}
        route_sizes = {k: 0 for k in expert_ids}
        predictions = np.empty(len(X_test), dtype=int)
        modulation_strengths = []

        for start in range(0, len(X_test), block_size):
            end = min(start + block_size, len(X_test))
            current_routing = routing_vectors[start:end]

            # Causal boundary: route and predict before updating state.
            if adaptive_router:
                distances = np.linalg.norm(
                    current_routing[:, None, :] - centers[None, :, :], axis=2)
                routes = distances.argmin(axis=1)
            else:
                routes = result["kmeans_model"].predict(current_routing)

            for expert_id in expert_ids:
                mask = routes == expert_id
                if not mask.any():
                    continue
                raw = X_test[start:end][mask]
                transformed = raw

                if modulation and feature_sizes[expert_id] >= minimum_history:
                    history = np.concatenate(list(feature_history[expert_id]))
                    current_median = np.median(history, axis=0)
                    current_scale = np.maximum(
                        (np.quantile(history, .75, axis=0) -
                         np.quantile(history, .25, axis=0)) / 1.349,
                        1e-9,
                    )
                    train_median, train_scale, lower, upper = references[expert_id]
                    drift = np.median(
                        np.abs(current_median - train_median) / train_scale)
                    strength = min(
                        maximum_modulation, drift / (1.0 + drift))
                    aligned = train_median + (
                        raw - current_median) * (train_scale / current_scale)
                    aligned = np.clip(aligned, lower, upper)
                    transformed = (
                        (1.0 - strength) * raw + strength * aligned)
                    modulation_strengths.append(strength)

                predictions[start:end][mask] = _predict_aligned(
                    experts[expert_id], transformed, number_of_classes)

            # The already-predicted unlabeled block may affect future blocks.
            for expert_id in expert_ids:
                mask = routes == expert_id
                if not mask.any():
                    continue
                feature_history[expert_id].append(X_test[start:end][mask])
                route_history[expert_id].append(current_routing[mask])
                feature_sizes[expert_id] += int(mask.sum())
                route_sizes[expert_id] += int(mask.sum())
                while (feature_sizes[expert_id] > buffer_size and
                       len(feature_history[expert_id]) > 1):
                    feature_sizes[expert_id] -= len(
                        feature_history[expert_id].popleft())
                while (route_sizes[expert_id] > buffer_size and
                       len(route_history[expert_id]) > 1):
                    route_sizes[expert_id] -= len(
                        route_history[expert_id].popleft())

            if adaptive_router:
                for expert_id in expert_ids:
                    if route_sizes[expert_id] < minimum_history:
                        continue
                    recent_center = np.mean(
                        np.concatenate(list(route_history[expert_id])), axis=0)
                    centers[expert_id] = (
                        (1.0 - center_learning_rate - center_anchor) *
                        centers[expert_id]
                        + center_learning_rate * recent_center
                        + center_anchor * original_centers[expert_id]
                    )

        return predictions, (
            float(np.mean(modulation_strengths))
            if modulation_strengths else 0.0)

    configurations = [
        ("hard_frozen", False, False),
        ("adaptive_router", True, False),
        ("frozen_modulation", False, True),
        ("adaptive_modulation", True, True),
    ]
    prediction_sets = {}
    strength_sets = {}
    for name, adaptive_router, modulation in configurations:
        prediction_sets[name], strength_sets[name] = predict_variant(
            adaptive_router, modulation)

    # Evaluation boundary: labels first become visible here.
    y_test = test[result["target_column"]].to_numpy()
    rows = []
    for name, predictions in prediction_sets.items():
        rows.append({
            "method": name,
            "accuracy": accuracy_score(y_test, predictions),
            "precision_weighted": precision_score(
                y_test, predictions, average="weighted", zero_division=0),
            "recall_weighted": recall_score(
                y_test, predictions, average="weighted"),
            "f1_weighted": f1_score(
                y_test, predictions, average="weighted"),
            "f1_macro": f1_score(y_test, predictions, average="macro"),
            "mean_modulation_strength": strength_sets[name],
        })
    metrics = pd.DataFrame(rows)
    baseline = metrics.loc[
        metrics["method"] == "hard_frozen", "accuracy"].iloc[0]
    metrics["accuracy_gain_pp"] = 100 * (metrics["accuracy"] - baseline)
    print(f"\n{scenario}")
    display(metrics.round(4))
    return {"base_result": result, "metrics": metrics,
            "predictions": prediction_sets}


In [ ]:
scenario_4_modulation = evaluate_adaptive_modulation(
    "scenario_4",
    "DATASETS/CDR-MLC/scale_1/Short/CDR-MLC-Shuffle.csv",
    "DATASETS/CDR-MLC/scale_1/Long/CDR-MLC-Shuffle.csv",
)


In [ ]:
scenario_5_modulation = evaluate_adaptive_modulation(
    "scenario_5",
    "DATASETS/CDR-MLC/scale_1/Long/CDR-MLC-Shuffle.csv",
    "DATASETS/CDR-MLC/scale_1/Short/CDR-MLC-Shuffle.csv",
)
